### tokenize(): lexical analysis — turn one expression string into a list of tokens
- scans the string left to right, one character at a time.
- produces `(type, value)` tuples for the parser: `NUM`, `OP`, `LPAREN`, `RPAREN`.
- a number literal is one or more digits, with an optional single `.` followed by one or more digits.
- the minus sign is kept as its own `OP` token (not folded into a number), so the parser can treat it as unary.
- raises `ValueError` on an unexpected character or a malformed number such as `3.`; the caller reports `ERROR`.

In [2]:
def tokenize(expression: str) -> list:
    """Break one expression string into a list of (type, value) tokens.

    Token types are NUM, OP, LPAREN and RPAREN. The function walks the string
    with an index and groups consecutive digits (and an optional decimal part)
    into a single NUM token. It raises ValueError when it meets a character
    that is not part of the grammar, or a number with a dot but no digits
    after it.
    """
    tokens = []
    i = 0
    length = len(expression)

    while i < length:
        char = expression[i]

        if char == ' ' or char == '\t':          # skip spaces and tabs
            i += 1

        elif char in '+-*/%^':                    # any operator symbol
            tokens.append(("OP", char))
            i += 1

        elif char == '(':
            tokens.append(("LPAREN", "("))
            i += 1

        elif char == ')':
            tokens.append(("RPAREN", ")"))
            i += 1

        elif char.isdigit():                      # read a whole number literal
            start = i
            while i < length and expression[i].isdigit():
                i += 1
            # a number may have a single dot followed by one or more digits
            if i < length and expression[i] == '.':
                i += 1
                if i >= length or not expression[i].isdigit():
                    raise ValueError("a number needs digits after the '.'")
                while i < length and expression[i].isdigit():
                    i += 1
            tokens.append(("NUM", float(expression[start:i])))

        else:                                     # anything else is not allowed
            raise ValueError("invalid character: " + char)

    return tokens

### show the tokens produced for a few sample expressions

In [3]:
for e in ["3 + 5", "2 + 3 * 4", "-(3 + 4)", "3.5 + 1.5", "3 @ 5"]:
    try:
        print(e, "->", tokenize(e))
    except ValueError as err:
        print(e, "-> ERROR (", err, ")")

3 + 5 -> [('NUM', 3.0), ('OP', '+'), ('NUM', 5.0)]
2 + 3 * 4 -> [('NUM', 2.0), ('OP', '+'), ('NUM', 3.0), ('OP', '*'), ('NUM', 4.0)]
-(3 + 4) -> [('OP', '-'), ('LPAREN', '('), ('NUM', 3.0), ('OP', '+'), ('NUM', 4.0), ('RPAREN', ')')]
3.5 + 1.5 -> [('NUM', 3.5), ('OP', '+'), ('NUM', 1.5)]
3 @ 5 -> ERROR ( invalid character: @ )


## Recursive-descent parser

### Parser Function

The `parse()` function converts a list of lexical tokens into a **parse tree (Abstract Syntax Tree)** representing the structure of an arithmetic expression. It uses a recursive-descent parsing approach, where separate helper functions handle different levels of operator precedence.

The parser processes expressions according to the following precedence and associativity rules:

1. **Addition and subtraction (`+`, `-`)** – left-associative.
2. **Multiplication, division, and modulo (`*`, `/`, `%`)** – left-associative. It also supports **implicit multiplication**, where a factor is directly followed by parentheses, such as `2(3 + 4)`.
3. **Unary minus (`-`)** – treated as a prefix operator and supports expressions such as `-5` or `-(2 + 3)`. Unary plus (`+`) is explicitly unsupported.
4. **Exponentiation (`^`)** – right-associative, with the exponent allowed to contain a unary minus, such as `2^-3`.
5. **Primary expressions** – numeric literals and parenthesised expressions.

The helper functions `peek()`, `advance()`, and `expect()` manage token inspection and consumption while tracking the current parsing position. Each parsing function returns a tree node representing the corresponding operation or value.

After parsing the complete expression, the function checks for any remaining tokens. If unexpected tokens, mismatched parentheses, unsupported operators, or other syntax errors are encountered, a `ValueError` is raised. Otherwise, the completed parse tree is returned.

In [4]:

def parse(tokens: list):
    """Parse a token list into a parse tree. Raises ValueError on any
    syntax error (including leftover tokens after a complete expression).
    """
    pos = 0

    def peek():
        return tokens[pos]

    def advance():
        nonlocal pos
        tok = tokens[pos]
        pos += 1
        return tok

    def expect(token_type):
        nonlocal pos
        tok = tokens[pos]
        if tok[0] != token_type:
            raise ValueError(f"Expected {token_type} but found {tok[0]}")
        pos += 1
        return tok

    # Level 1: + - (left associative)
    def parse_expression():
        node = parse_term()
        while peek()[0] == 'OP' and peek()[1] in ('+', '-'):
            op = advance()[1]
            right = parse_term()
            node = ('binop', op, node, right)
        return node

    # Level 2: * / % and implicit multiplication (left associative)
    def parse_term():
        node = parse_unary()
        while True:
            tok = peek()
            if tok[0] == 'OP' and tok[1] in ('*', '/', '%'):
                op = advance()[1]
                right = parse_unary()
                node = ('binop', op, node, right)
            elif tok[0] == 'LPAREN':
                # Implicit multiplication: a factor directly followed by '('
                right = parse_unary()
                node = ('binop', '*', node, right)
            else:
                break
        return node

    # Level 3: unary minus (prefix). Unary '+' is explicitly unsupported.
    def parse_unary():
        tok = peek()
        if tok[0] == 'OP' and tok[1] == '-':
            advance()
            operand = parse_unary()
            return ('neg', operand)
        if tok[0] == 'OP' and tok[1] == '+':
            raise ValueError("Unary '+' is not supported")
        return parse_power()

    # Level 4: ^ (right associative; exponent may itself carry a unary '-')
    def parse_power():
        base = parse_primary()
        tok = peek()
        if tok[0] == 'OP' and tok[1] == '^':
            advance()
            exponent = parse_unary()
            return ('binop', '^', base, exponent)
        return base

    # Numbers and parenthesised sub-expressions
    def parse_primary():
        tok = peek()
        if tok[0] == 'NUM':
            advance()
            return ('num', float(tok[1]))
        if tok[0] == 'LPAREN':
            advance()
            node = parse_expression()
            expect('RPAREN')
            return node
        raise ValueError(f"Unexpected token {tok[0]}")

    tree = parse_expression()
    if peek()[0] != 'END':
        raise ValueError(f"Unexpected trailing token {peek()[0]}")
    return tree